# Can you recover a planet from public velocities? Fit iota Dra b's eccentric orbit

[Open this notebook in Colab](https://colab.research.google.com/github/Rakiura-Inc/limits-of-habitability/blob/main/notebooks/gaia_03_inner_planet_from_velocities.ipynb)

The lab's paper on iota Draconis fit two decades of velocities and found two companions: iota Dra b, on an eccentric 511-day orbit, and iota Dra c, much further out ([Hill et al. 2021](https://arxiv.org/abs/2107.13583)). In this notebook you build orbitize!'s input from the public velocities, fit both companions to the velocities alone, and check b against the paper. This is the starting point for the joint fit with Hipparcos and Gaia.

Run the cells in order (Shift+Enter). Boxes headed **Write down** ask you questions. Answer each one in the cell underneath by replacing its `...`, then run that cell: the notebook keeps your answer. The last cell puts all your answers into a single table and saves it with your plots. That table is what you hand in on the step.

In [ ]:
# Setup. Run this first. In Colab it installs what the notebook needs.
# orbitize! is built from source in Colab, so this takes about two minutes the first time.
# If Colab asks you to restart the session afterwards, restart it and run the notebook from the top.
import importlib.util

# habtrain: a small package written for these notebooks. It downloads the data, draws some of the
# plots and records your answers. The science calls are in the cells, where you can read them.
if importlib.util.find_spec("habtrain") is None or importlib.util.find_spec("orbitize") is None:
    %pip install -q "habtrain[orbits] @ git+https://github.com/Rakiura-Inc/limits-of-habitability@main"

# rakiura: the optional helper that connects this notebook to your step in Rakiura.
if importlib.util.find_spec("rakiura") is None:
    %pip install -q "rakiura @ git+https://github.com/Rakiura-Inc/rakiura-py@main"

import habtrain as ht  # the notebooks' package, used as ht.data, ht.fit, ht.plot and so on
import matplotlib.pyplot as plt  # plots
import numpy as np  # arrays and arithmetic

# Which notebook this is, so your hand-in records exactly what you ran.
NOTEBOOK = "notebooks/gaia_03_inner_planet_from_velocities.ipynb"
MODE = ht.portal.connect(NOTEBOOK)

> **Connect to Rakiura (optional)**
>
> Connect, and Rakiura ticks off each part as you finish it, with your plots on the step. You still attach your answers table and press Hand in on the step yourself. Skip it and everything still works: you attach the saved files on the step.
>
> To connect, run the cell below, open the link it prints, check the code matches the one shown, and click Allow. In Colab you do this once per session. To skip the click next time, add your key as a Colab secret named `RAKIURA_KEY` (the key icon on the left; your key is in Rakiura under Settings > Notebook key).

In [ ]:
# Connect to Rakiura (optional). Open the link it prints, check the code matches, click Allow.
MODE = ht.portal.connect(NOTEBOOK, sign_in=True)

## The velocities

Two public tables hold most of the paper's velocities:
- **Lick Observatory, 2000 to 2005:** 119 velocities ([Butler et al. 2006](https://arxiv.org/abs/astro-ph/0607493), VizieR J/ApJ/646/505)
- **The Automated Planet Finder (APF), 2018 to 2021:** 456 velocities (Hill et al. 2021, VizieR J/AJ/162/211)

The paper also used 46 more Lick velocities, taken between these two sets, that are not in a public table, so your fit will not match the paper exactly. orbitize! wants velocities in km/s and dates as MJD (Julian date minus 2,400,000.5), with a label for each telescope, so the cell below converts both tables.

In [ ]:
import pandas as pd

# The two public tables of iota Dra's velocities as published: Julian dates, velocities in m/s.
public = ht.orbits.iota_dra_tables()

# orbitize! reads dates as MJD and velocities in km/s,
# with object 0 (the star) and a telescope label.
rv = pd.DataFrame({
    "epoch": public.jd - ht.orbits.MJD_OFFSET,  # MJD = JD - 2400000.5
    "object": 0,
    "rv": public.rv_ms / 1000,
    "rv_err": public.error_ms / 1000,
    "instrument": public.telescope,
})
rv = rv.sort_values("epoch").reset_index(drop=True)
rv.to_csv("iota_dra_rv.csv", index=False, float_format="%.6f")
print(rv.instrument.value_counts().to_string())

In [ ]:
# The velocities over time, one colour per telescope.
fig_rv, ax = plt.subplots(figsize=(10, 4), dpi=110)
for telescope, colour in (("lick", "#3b6ea5"), ("apf", "#d1495b")):
    mine = rv[rv.instrument == telescope]
    # orbitize!'s table is in km/s; the plot shows m/s
    ax.errorbar(
        mine.epoch,
        mine.rv * 1000,
        yerr=mine.rv_err * 1000,
        fmt="o",
        ms=3,
        color=colour,
        label=telescope.upper(),
    )
ax.set_xlabel("Date (MJD)")
ax.set_ylabel("Velocity (m/s)")
ax.legend()
ax.set_title("iota Dra: every public velocity", fontsize=11)
credit = (
    "Data: Butler et al. 2006 (Lick); Hill et al. 2021 (APF). VizieR J/ApJ/646/505, J/AJ/162/211."
)
fig_rv.text(0.01, 0.005, credit, fontsize=7, color="#666")
ht.record.keep_figure("inner-planet-fitted", "velocities", fig_rv)
plt.show()

Look for the gap of more than twelve years between the two telescopes, and the sharp swings: b's orbit is so stretched that the star's velocity changes fast near closest approach and slowly the rest of the time.

## An eccentric orbit

The bootcamp fitted circles. An eccentric orbit needs four numbers for its shape and timing, besides the size of the swing:
- **Period:** how long one orbit takes.
- **Eccentricity (e):** how stretched it is, from 0 for a circle towards 1.
- **Argument of periastron (ω):** where in the orbit the closest approach (periastron) happens, as an angle.
- **Time of periastron:** when the companion passes closest. orbitize! stores it as a fraction of the period, called tau.

orbitize! describes each orbit by its **semi-major axis** (half the long axis, in AU) instead of its period, and fits the companion's mass directly. With the orbit held edge on (inclination 90 degrees), the mass it fits from velocities is the minimum mass.

Two traps:
- Velocity papers quote the **star's** argument of periastron. orbitize! uses the **companion's**, which is the star's plus 180 degrees (π radians).
- orbitize! works in radians, km/s and MJD.

## Why fit in two stages

A fit with 20 numbers free, started anywhere, can wander off: in testing, a free search found a negative mass and, another time, an eccentricity of 0.58 for b. So fit in two stages:
1. Hold b's shape at the paper's values and find c and the two telescopes' offsets, trying eight different starting guesses for c and keeping the best.
2. Start from that answer, free b's shape and the two jitters, and fit everything together.

Each stage finds the single most likely orbit (a best fit), not a posterior. The joint fit in the next step starts its walkers in a small ball around this answer. Together the stages take a few minutes, up to ten.

The set-up follows orbitize!'s [Radial Velocity Tutorial for MCMC](https://orbitize.readthedocs.io/en/latest/tutorials/RV_MCMC_Tutorial.html) (BSD 3-Clause licence, kept in `LICENSES/orbitize.txt`): the data table, a `System`, its parameter labels, and orbitize!'s own model and likelihood. Where that tutorial starts the walkers straight away, this notebook first finds the best fit they start from.

In [ ]:
from orbitize import read_input, system

# The system: iota Dra with two companions (b first, then c), velocities only.
star = ht.orbits.IOTA_DRA  # mass from asteroseismology and parallax from Gaia, as the paper uses
iota_dra = system.System(
    num_secondary_bodies=2,  # b and c
    data_table=read_input.read_file("iota_dra_rv.csv"),
    stellar_or_system_mass=star["star_mass"],  # solar masses
    plx=star["parallax"],  # milliarcseconds
    mass_err=star["star_mass_err"],
    plx_err=star["parallax_err"],
    fit_secondary_mass=True,  # fit the companions' masses, which velocities measure
)
index = iota_dra.param_idx  # each parameter's position in orbitize!'s list, by name
print(iota_dra.labels)

In [ ]:
from orbitize import basis

# Start: b at the paper's orbit, seen edge on, so its fitted mass is its minimum mass. c is unknown.
mjup = ht.orbits.MJUP_IN_MSUN
published = ht.orbits.published_iota_dra_b().set_index("quantity").value
periastron_mjd = ht.orbits.B_PERIASTRON_JD - ht.orbits.MJD_OFFSET
start_values = {
    "sma1": published["semi-major axis (AU)"],
    "ecc1": published["eccentricity"],
    "inc1": np.pi / 2,  # edge on
    "aop1": ht.orbits.B_OMEGA_STAR + np.pi,  # the paper gives the star's angle; orbitize! wants b's
    "pan1": 1.0,  # the orbit's angle on the sky: velocities cannot tell, so any value will do
    "tau1": basis.tp_to_tau(periastron_mjd, iota_dra.tau_ref_epoch, ht.orbits.B_PERIOD_DAYS),
    "inc2": np.pi / 2,
    "pan2": 1.0,
    "plx": star["parallax"],
    "m0": star["star_mass"],
    "m1": published["minimum mass (Jupiter masses)"] * mjup,
    "sigma_apf": 0.011,  # km/s, about the published jitters
    "sigma_lick": 0.014,
}
start = np.zeros(len(iota_dra.labels))  # everything else, c included, starts at 0
for name, value in start_values.items():
    start[index[name]] = value

In [ ]:
from orbitize import lnlike

# How well one set of parameters fits the velocities: orbitize!'s model, then its likelihood.
# orbitize! keeps two columns for each measurement; for a velocity the second is empty (nan).
table = iota_dra.data_table
measured = np.array([table["quant1"], table["quant2"]]).T
errors = np.array([table["quant1_err"], table["quant2_err"]]).T


def log_likelihood(params):
    model, jitter = iota_dra.compute_model(params)
    per_point = lnlike.chi2_lnlike(measured, errors, None, model, jitter, iota_dra.all_seppa)
    return np.nansum(per_point)  # nansum skips the empty second column

In [ ]:
REFUSED = 1e12  # the cost of an orbit that cannot be real: far worse than any real one


def with_values(start, free, values):
    """A copy of `start` with the parameters named in `free` set to `values`, in the same order."""
    params = start.copy()
    for name, value in zip(free, values, strict=True):
        params[index[name]] = value
    return params


def possible(params):
    """False for an orbit that cannot be real, or that puts c where the notebook does not look."""
    eccentricities_ok = 0 <= params[index["ecc1"]] < 0.95 and 0 <= params[index["ecc2"]] < 0.95
    masses_ok = params[index["m1"]] > 0 and params[index["m2"]] > 0
    c_in_range = 3 < params[index["sma2"]] < 100  # AU
    return eccentricities_ok and masses_ok and c_in_range

In [ ]:
from scipy.optimize import minimize


def cost(values, start, free):
    """What Nelder-Mead makes as small as it can: minus the log likelihood."""
    params = with_values(start, free, values)
    if not possible(params):
        return REFUSED
    return -float(log_likelihood(params))


def best_fit(start, first_guess):
    """The most likely orbit, moving only the parameters named in `first_guess` from `start`.

    `first_guess` maps each free parameter's name to where the search starts for it.
    Returns (its log likelihood, all its parameters).
    """
    free = list(first_guess)
    # Nelder-Mead needs no derivatives. adaptive suits a search over this many parameters; the
    # tolerances stop it once the orbit stops changing.
    options = {"maxiter": 20000, "xatol": 1e-8, "fatol": 1e-5, "adaptive": True}
    guess = list(first_guess.values())
    result = minimize(cost, guess, args=(start, free), method="Nelder-Mead", options=options)
    return -float(result.fun), with_values(start, free, result.x)

In [ ]:
def orbit_of_b(params):
    """b's orbit and the two jitters, in the units the paper reports."""
    return {
        "semi-major axis (AU)": round(float(params[index["sma1"]]), 4),
        "eccentricity": round(float(params[index["ecc1"]]), 4),
        "minimum mass (Jupiter masses)": round(float(params[index["m1"]] / mjup), 2),
        "APF jitter (m/s)": round(float(params[index["sigma_apf"]]) * 1000, 1),
        "Lick jitter (m/s)": round(float(params[index["sigma_lick"]]) * 1000, 1),
    }

In [ ]:
# Stage 1, b's shape held. Free: the two offsets (gamma), c's orbit and mass, and b's timing.
# Velocities barely pin down c, so one starting guess can stop in a poor fit. Try eight random
# guesses for c and keep the best. The seed makes your run match the worked answer.
rng = np.random.default_rng(seed=2)
best_log_likelihood = -np.inf
for _ in range(8):
    guess = {
        "gamma_apf": 0,  # km/s
        "gamma_lick": 0,
        "sma2": rng.uniform(8, 40),  # AU
        "ecc2": rng.uniform(0.1, 0.7),
        "aop2": rng.uniform(0, 2 * np.pi),  # radians
        "tau2": rng.uniform(0, 1),
        "m2": rng.uniform(5, 40) * mjup,  # 5 to 40 Jupiter masses
        "tau1": start[index["tau1"]],
        "aop1": start[index["aop1"]],
    }
    log_likelihood_here, params = best_fit(start, guess)
    if log_likelihood_here > best_log_likelihood:
        best_log_likelihood = log_likelihood_here
        stage1 = params

In [ ]:
print("Stage 1, b held:", orbit_of_b(stage1))
# If c prints at 100.0 AU, the search stopped at the edge possible() allows: these velocities
# cannot place c, which WHY_C_IS_LOOSE below asks about.
c_sma_au = stage1[index["sma2"]]
c_mass_jupiter = stage1[index["m2"]] / mjup
print(f"c: {c_sma_au:.1f} AU, minimum mass {c_mass_jupiter:.1f} Jupiter masses")

If c prints at 100.0 AU, the search stopped at its outer edge. These velocities cannot tell where c is, only that it is far out. Hipparcos and Gaia pin it down in the next step.

In [ ]:
# Stage 2: start from stage 1 and free b's size, shape and minimum mass, and both jitters.
stage1_free = ["gamma_apf", "gamma_lick", "sma2", "ecc2", "aop2", "tau2", "m2", "tau1", "aop1"]
free_stage2 = [*stage1_free, "ecc1", "m1", "sma1", "sigma_apf", "sigma_lick"]
stage2_guess = {name: stage1[index[name]] for name in free_stage2}
log_likelihood_stage2, stage2 = best_fit(stage1, stage2_guess)
my_b = orbit_of_b(stage2)
print("Stage 2, everything free:", my_b)

# Sanity check: b must come back near eccentricity 0.70.
if abs(my_b["eccentricity"] - 0.70) > 0.05:
    print(
        "b's eccentricity has moved far from 0.70: the fit has found a different orbit. "
        "Run stage 1 again with more attempts."
    )

## Check against the paper

The table below puts your stage 2 numbers next to the paper's table 3, which fit the same kind of model with RadVel, a different code. Your numbers are a best fit with no error bars, so compare them with the paper's range: how many of the paper's sigmas away is each one?

In [ ]:
import pandas as pd

# Your numbers against the paper's, and how many of
# the paper's sigmas away (yours has no error bar).
rows = []
for paper in ht.orbits.published_iota_dra_b().itertuples():
    yours = my_b[paper.quantity]
    if yours > paper.value:
        # above the paper's value: its upper error bar
        sigmas_away = (yours - paper.value) / paper.plus
    else:
        sigmas_away = (yours - paper.value) / paper.minus
    rows.append({
        "": paper.quantity,
        "yours": yours,
        "paper": f"{paper.value} (-{paper.minus}, +{paper.plus})",
        "paper's sigmas away": round(sigmas_away, 1),
    })
comparison = pd.DataFrame(rows)
comparison

> **Write down: how your fit compares**
>
> - Which of b's numbers land within 1 sigma of the paper, and which do not? One line. Put it in `MATCH`.
> - Why might the Lick jitter differ more than the rest? One line. Put it in `LICK_DIFFERS`.
> - Why is c barely pinned down by these velocities? Look at its period against the time the data covers. One or two lines. Put it in `WHY_C_IS_LOOSE`.
>
> **Where to look:** the comparison table, the velocity plot, and the note on the missing Lick velocities.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

<details><summary>Stuck? Open hint 1</summary>

c's orbit is many decades long. Compare that with the span from the first Lick velocity to the last APF one.

</details>

In [ ]:
MATCH = ...  # one line, in quotes
LICK_DIFFERS = ...  # one line, in quotes
WHY_C_IS_LOOSE = ...  # one or two lines, in quotes

# Save the answers, with your fitted numbers and the comparison.
ht.record.save("inner_planet", my_b, "iota Dra b and the jitters, from your stage 2 fit")
ht.record.save("comparison", comparison, "your fit against Hill et al. 2021, table 3")
ht.record.save("match", MATCH, "which numbers match the paper")
ht.record.save("lick_differs", LICK_DIFFERS, "why the Lick jitter may differ")
ht.record.save("why_c_is_loose", WHY_C_IS_LOOSE, "why c is barely pinned down by velocities")

### Hand in

Run the cell below. It shows all your answers in one table. Check them, then hand in this step in Rakiura, either by pasting the table in or by attaching the saved files. To get the files in Colab, click the folder icon on the left, open `handin`, then `inner-planet-fitted`, and download them.

In [ ]:
# Your answers and the velocity plot, saved as your hand-in.
ht.record.handin(
    "inner-planet-fitted", ["inner_planet", "comparison", "match", "lick_differs", "why_c_is_loose"]
)